In [1]:
import os, json, time, requests
import numpy as np
import deepchem as dc

DATA = os.path.expanduser("~/ai-nose/data")
OUT  = os.path.expanduser("~/ai-nose/llm_eval"); os.makedirs(OUT, exist_ok=True)
train = dc.data.DiskDataset(f"{DATA}/train")
test  = dc.data.DiskDataset(f"{DATA}/test")
TASKS = list(test.tasks)
gnn = np.load(os.path.expanduser("~/ai-nose/models/ensemble/test_ensemble_probs.npy"))

MODEL = "qwen3:4b-instruct"
CACHE = f"{OUT}/{MODEL.replace(':', '_')}.jsonl"

# Forced output: exactly 5 words, each one from the 138-word list
SCHEMA = {"type": "object",
          "properties": {"smells": {"type": "array",
                                    "items": {"type": "string", "enum": TASKS},
                                    "minItems": 5, "maxItems": 5}},
          "required": ["smells"]}

# The fixed part goes first (Ollama reuses it), the molecule goes last
INSTRUCTIONS = ("You are an expert perfumer and flavour chemist. "
                "You will be given a molecule as a SMILES string. "
                "Choose the 5 odor descriptors from the allowed list that best describe how it smells, "
                "most likely first. Use only words from the list.\n\n"
                "Allowed descriptors: " + ", ".join(TASKS) + "\n\n")

def ask_llm(smiles):
    body = {"model": MODEL, "stream": False, "format": SCHEMA,
            "options": {"temperature": 0, "seed": 0, "num_ctx": 2048},
            "messages": [{"role": "user", "content": INSTRUCTIONS + f"Molecule (SMILES): {smiles}"}]}
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=180)
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])["smells"]

# Load answers saved by earlier runs, so we never ask twice
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE):
        rec = json.loads(line); done[rec["smiles"]] = rec["smells"]

def run(n=None):
    ids = list(test.ids) if n is None else list(test.ids[:n])
    todo = [s for s in ids if s not in done]
    t0 = time.time()
    with open(CACHE, "a") as f:
        for k, s in enumerate(todo, 1):
            try:
                smells = ask_llm(s)
            except Exception as e:
                print("error (will retry next run):", s, e); continue
            done[s] = smells
            f.write(json.dumps({"smiles": s, "smells": smells}) + "\n"); f.flush()
            if k % 25 == 0 or k == len(todo):
                print(f"{k}/{len(todo)} done, {(time.time() - t0) / k:.1f}s per molecule", flush=True)

def top5(p):
    return [TASKS[j] for j in np.argsort(p)[::-1][:5]]

common5 = [TASKS[j] for j in np.argsort(train.y.sum(axis=0))[::-1][:5]]   # 5 most common smells in training

def score(picks_list, idx):
    prec, rec, hit1 = [], [], []
    for i, picks in zip(idx, picks_list):
        true = {TASKS[j] for j in np.where(test.y[i] == 1)[0]}
        uniq = list(dict.fromkeys(picks))[:5]          # drop repeated words
        correct = len(set(uniq) & true)
        prec.append(correct / 5)
        rec.append(correct / len(true))
        hit1.append(bool(uniq) and uniq[0] in true)
    return np.mean(prec), np.mean(rec), np.mean(hit1)

def report():
    idx = [i for i, s in enumerate(test.ids) if s in done]
    rng = np.random.default_rng(0)
    rows = {f"LLM ({MODEL})":                    [done[test.ids[i]] for i in idx],
            "GNN ensemble (yours)":              [top5(gnn[i]) for i in idx],
            "Always the 5 most common smells":   [common5 for _ in idx],
            "Random 5 smells":                   [list(rng.choice(TASKS, 5, replace=False)) for _ in idx]}
    print(f"Scored on {len(idx)} test molecules\n")
    print(f"{'method':36s} precision@5  recall@5  first pick right")
    for name, picks in rows.items():
        p, r, h = score(picks, idx)
        print(f"{name:36s} {p:11.3f} {r:9.3f} {h:16.0%}")

print("Ready. 5 most common smells:", common5)

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
No normalization for NumAmideBonds. Feature removed!
No normalization for NumAtomStereoCenters. Feature removed!
No normalization for NumBridgeheadAtoms. Feature removed!
No normalization for NumHeterocycles. Feature removed!
No normalization for NumSpiroAtoms. Feature removed!
No normalization for NumUnspecifiedAtomStereoCenters. Feature removed!
No normalization for Phi. Feature removed!
Skipped loading material featurizer, missing a dependency. No module named 'ase'
/home/sachinkukkar/miniforge3/envs/open_pom/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading some PyTorch models, missing a dependency. No module

Ready. 5 most common smells: ['fruity', 'green', 'sweet', 'floral', 'herbal']


In [3]:
run(10)
for i in range(5):
    s = test.ids[i]
    true = [TASKS[j] for j in np.where(test.y[i] == 1)[0]]
    print(f"\n{s}\n  real: {', '.join(true)}\n  LLM:  {', '.join(done[s])}\n  GNN:  {', '.join(top5(gnn[i]))}")
print()
report()


CCCC=O
  real: chocolate, cocoa, fruity, green, malty, musty, pungent
  LLM:  fruity, sweet, clean, ketonic, apple
  GNN:  fruity, ethereal, pungent, green, nutty

O=CO
  real: fermented, fruity, pungent, sharp
  LLM:  alcoholic, fruity, sweet, clean, caramellic
  GNN:  odorless, sour, sharp, pungent, ethereal

Cc1cccc(O)c1
  real: dry, leathery, medicinal, phenolic, woody
  LLM:  cabbage, phenolic, medicinal, sharp, earthy
  GNN:  phenolic, medicinal, woody, smoky, animal

O=C(O)C(O)C(O)C(=O)O
  real: caramellic
  LLM:  apple, fruity, fresh, juicy, clean
  GNN:  odorless, sweet, sour, buttery, caramellic

O=C(O)C(=O)CO
  real: burnt, buttery, cabbage, creamy, radish, sour
  LLM:  fruity, sweet, creamy, almond, buttery
  GNN:  caramellic, odorless, burnt, sour, sweet

Scored on 10 test molecules

method                               precision@5  recall@5  first pick right
LLM (qwen3:4b-instruct)                    0.280     0.234              30%
GNN ensemble (yours)                  

In [6]:
# Prompt version 2: tell the model the exact JSON format that Ollama enforces
INSTRUCTIONS = ("You are an expert perfumer and flavour chemist. "
                "You will be given a molecule as a SMILES string. "
                "Choose the 5 odor descriptors from the allowed list that best describe how it smells, "
                "most likely first. Use only words from the list.\n\n"
                "Allowed descriptors: " + ", ".join(TASKS) + "\n\n"
                'Answer with JSON only, in exactly this format: '
                '{"smells": ["word1", "word2", "word3", "word4", "word5"]}\n\n')

# New answers file: every molecule must be asked with the same prompt
CACHE = f"{OUT}/{MODEL.replace(':', '_')}_v2.jsonl"
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE):
        rec = json.loads(line); done[rec["smiles"]] = rec["smells"]

def ask_llm(smiles):
    body = {"model": MODEL, "stream": False, "format": SCHEMA, "keep_alive": "30m",
            "options": {"temperature": 0, "seed": 0, "num_ctx": 1024, "num_predict": 120},
            "messages": [{"role": "user", "content": INSTRUCTIONS + f"Molecule (SMILES): {smiles}"}]}
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=120)
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])["smells"]

def run(n=None):
    ids = list(test.ids) if n is None else list(test.ids[:n])
    todo = [s for s in ids if s not in done]
    t0 = time.time()
    with open(CACHE, "a") as f:
        for k, s in enumerate(todo, 1):
            t1 = time.time()
            try:
                smells = ask_llm(s)
            except Exception as e:
                print(f"{k}/{len(todo)} error (will retry next run): {e}", flush=True); continue
            done[s] = smells
            f.write(json.dumps({"smiles": s, "smells": smells}) + "\n"); f.flush()
            if n is not None or k <= 5 or k % 25 == 0 or k == len(todo):
                print(f"{k}/{len(todo)} done in {time.time() - t1:.1f}s (average {(time.time() - t0) / k:.1f}s)", flush=True)

print("Answers saved so far (v2 prompt):", len(done))
run(10)   # quick check before the full run

Answers saved so far (v2 prompt): 0
1/10 done in 1.4s (average 1.4s)
2/10 done in 1.2s (average 1.3s)
3/10 done in 1.0s (average 1.2s)
4/10 done in 1.0s (average 1.2s)
5/10 done in 1.0s (average 1.1s)
6/10 done in 1.1s (average 1.1s)
7/10 done in 1.1s (average 1.1s)
8/10 done in 1.1s (average 1.1s)
9/10 done in 1.1s (average 1.1s)
10/10 done in 1.1s (average 1.1s)


In [7]:
run()

1/485 done in 1.4s (average 1.4s)
2/485 done in 1.1s (average 1.2s)
3/485 done in 1.0s (average 1.2s)
4/485 done in 0.9s (average 1.1s)
5/485 done in 1.0s (average 1.1s)
25/485 done in 1.1s (average 1.1s)
50/485 done in 0.8s (average 1.1s)
75/485 done in 0.9s (average 1.0s)
100/485 done in 1.0s (average 1.0s)
125/485 done in 0.9s (average 1.0s)
150/485 done in 1.0s (average 1.0s)
175/485 done in 0.9s (average 1.0s)
200/485 done in 1.0s (average 1.0s)
225/485 done in 1.0s (average 1.0s)
250/485 done in 0.8s (average 1.0s)
275/485 done in 0.9s (average 1.0s)
300/485 done in 1.1s (average 1.0s)
325/485 done in 1.2s (average 1.0s)
350/485 done in 1.3s (average 1.0s)
375/485 done in 1.1s (average 1.0s)
400/485 done in 1.1s (average 1.0s)
425/485 done in 1.1s (average 1.0s)
450/485 done in 1.0s (average 1.0s)
475/485 done in 1.2s (average 1.1s)
485/485 done in 1.1s (average 1.1s)


In [8]:
from collections import Counter

def per_molecule(picks_list):
    prec, rec, hit1 = [], [], []
    for i, picks in enumerate(picks_list):
        true = {TASKS[j] for j in np.where(test.y[i] == 1)[0]}
        uniq = list(dict.fromkeys(picks))[:5]
        correct = len(set(uniq) & true)
        prec.append(correct / 5); rec.append(correct / len(true))
        hit1.append(float(bool(uniq) and uniq[0] in true))
    return np.array(prec), np.array(rec), np.array(hit1)

n = len(test.ids)
failed = [s for s in test.ids if s not in done]
print(f"LLM answered {n - len(failed)} of {n} molecules ({len(failed)} failures, counted as wrong)\n")

rng = np.random.default_rng(0)
methods = {f"LLM ({MODEL})":                   [done.get(s, []) for s in test.ids],
           "GNN ensemble (yours)":             [top5(gnn[i]) for i in range(n)],
           "Always the 5 most common smells":  [common5] * n,
           "Random 5 smells":                  [list(rng.choice(TASKS, 5, replace=False)) for _ in range(n)]}
scores = {name: per_molecule(p) for name, p in methods.items()}

print(f"{'method':36s} precision@5  recall@5  first pick right")
for name, (p, r, h) in scores.items():
    print(f"{name:36s} {p.mean():11.3f} {r.mean():9.3f} {h.mean():16.0%}")

# Paired bootstrap: is the difference real, or could it be luck?
def diff_ci(a, b, reps=2000):
    idx = rng.integers(0, len(a), (reps, len(a)))
    d = a[idx].mean(1) - b[idx].mean(1)
    return d.mean(), np.percentile(d, 2.5), np.percentile(d, 97.5)

llm_p = scores[f"LLM ({MODEL})"][0]
for other in ["Always the 5 most common smells", "GNN ensemble (yours)"]:
    m, lo, hi = diff_ci(llm_p, scores[other][0])
    print(f"\nLLM minus {other}: precision@5 {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

# Which words does the LLM reach for, compared with what's actually common?
llm_words  = Counter(w for s in test.ids for w in dict.fromkeys(done.get(s, [])))
true_words = Counter(TASKS[j] for i in range(n) for j in np.where(test.y[i] == 1)[0])
print("\nLLM's most-used words:  ", ", ".join(f"{w} ({c})" for w, c in llm_words.most_common(8)))
print("Most common real smells:", ", ".join(f"{w} ({c})" for w, c in true_words.most_common(8)))

LLM answered 495 of 495 molecules (0 failures, counted as wrong)

method                               precision@5  recall@5  first pick right
LLM (qwen3:4b-instruct)                    0.193     0.183              38%
GNN ensemble (yours)                       0.480     0.500              72%
Always the 5 most common smells            0.275     0.255              39%
Random 5 smells                            0.039     0.037               5%

LLM minus Always the 5 most common smells: precision@5 -0.082 (95% CI -0.099 to -0.066)

LLM minus GNN ensemble (yours): precision@5 -0.287 (95% CI -0.307 to -0.266)

LLM's most-used words:   fruity (461), sweet (339), creamy (229), banana (177), green (139), nutty (137), dairy (127), woody (116)
Most common real smells: fruity (191), green (152), sweet (144), floral (114), herbal (80), woody (66), fatty (60), fresh (53)


In [9]:
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator
from sklearn.neighbors import NearestNeighbors

# 1. Retriever: 5 most similar TRAINING molecules by fingerprint (no GNN involved)
gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
bits = lambda smiles: np.array([gen.GetFingerprintAsNumPy(Chem.MolFromSmiles(s)) for s in smiles]).astype(bool)
nn = NearestNeighbors(n_neighbors=5, metric="jaccard").fit(bits(train.ids))
dist, nbr_idx = nn.kneighbors(bits(test.ids))
nbr_sim = 1 - dist
train_smells = [[TASKS[t] for t in np.where(row == 1)[0]] for row in train.y]

# 2. Prompt v3: same instructions, plus the neighbours as evidence (target molecule last)
def rag_prompt(i):
    lines = [f"- {train.ids[j]} smells: {', '.join(train_smells[j])}" for j in nbr_idx[i]]
    return (INSTRUCTIONS +
            "For reference, here are the 5 most structurally similar molecules from a perfume catalogue, "
            "with their known smells:\n" + "\n".join(lines) + "\n\n" +
            f"Molecule (SMILES): {test.ids[i]}")

CACHE_RAG = f"{OUT}/{MODEL.replace(':', '_')}_v3_rag.jsonl"
done_rag = {}
if os.path.exists(CACHE_RAG):
    for line in open(CACHE_RAG):
        rec = json.loads(line); done_rag[rec["smiles"]] = rec["smells"]

def ask_llm_prompt(prompt):
    body = {"model": MODEL, "stream": False, "format": SCHEMA, "keep_alive": "30m",
            "options": {"temperature": 0, "seed": 0, "num_ctx": 2048, "num_predict": 120},
            "messages": [{"role": "user", "content": prompt}]}
    r = requests.post("http://localhost:11434/api/chat", json=body, timeout=120)
    r.raise_for_status()
    return json.loads(r.json()["message"]["content"])["smells"]

def run_rag():
    todo = [i for i, s in enumerate(test.ids) if s not in done_rag]
    t0 = time.time()
    with open(CACHE_RAG, "a") as f:
        for k, i in enumerate(todo, 1):
            s = test.ids[i]
            try:
                smells = ask_llm_prompt(rag_prompt(i))
            except Exception as e:
                print(f"{k}/{len(todo)} error (will retry next run): {e}", flush=True); continue
            done_rag[s] = smells
            f.write(json.dumps({"smiles": s, "smells": smells}) + "\n"); f.flush()
            if k <= 3 or k % 50 == 0 or k == len(todo):
                print(f"{k}/{len(todo)} done (average {(time.time() - t0) / k:.1f}s)", flush=True)

print("End of an example prompt:\n...", rag_prompt(0)[-450:], "\n")
run_rag()

End of an example prompt:
... ume catalogue, with their known smells:
- CCCCC=O smells: berry, chocolate, fermented, fruity, nutty, pungent
- CCCCCC=O smells: aldehydic, clean, fatty, fresh, fruity, grassy, green, leafy, sweaty, vegetable, woody
- CCCC=CCCCCC=O smells: cucumber, fatty, green, herbal
- CCCCCCCCCCCCCC=O smells: amber, citrus, creamy, dairy, dry, fatty, fishy, fruit skin, fruity, musk, pear, waxy
- CCCCCCCCCCCCCCC=O smells: fresh, waxy

Molecule (SMILES): CCCC=O 

1/495 done (average 23.3s)
2/495 done (average 12.2s)
3/495 done (average 8.5s)
50/495 done (average 1.6s)
100/495 done (average 1.3s)
150/495 done (average 1.3s)
200/495 done (average 1.2s)
250/495 done (average 1.2s)
300/495 done (average 1.2s)
350/495 done (average 1.2s)
400/495 done (average 1.2s)
450/495 done (average 1.2s)
495/495 done (average 1.2s)


In [10]:
def copy_neighbours(i):
    # No LLM: rank smells by how often they appear among the 5 neighbours, weighted by similarity
    score = Counter()
    for j, sim in zip(nbr_idx[i], nbr_sim[i]):
        for w in train_smells[j]:
            score[w] += sim
    ranked = [w for w, _ in score.most_common()]
    ranked += [w for w in common5 if w not in ranked]   # fill up if the neighbours had < 5 smells
    return ranked[:5]

n = len(test.ids)
methods = {"GNN ensemble (yours)":             [top5(gnn[i]) for i in range(n)],
           "LLM + retrieved neighbours":       [done_rag.get(s, []) for s in test.ids],
           "Copy neighbours' smells (no LLM)": [copy_neighbours(i) for i in range(n)],
           "LLM zero-shot":                    [done.get(s, []) for s in test.ids],
           "Always the 5 most common smells":  [common5] * n}
scores = {name: per_molecule(p) for name, p in methods.items()}

print(f"Failures: {sum(s not in done_rag for s in test.ids)} (counted as wrong)\n")
print(f"{'method':36s} precision@5  recall@5  first pick right")
for name, (p, r, h) in scores.items():
    print(f"{name:36s} {p.mean():11.3f} {r.mean():9.3f} {h.mean():16.0%}")

rag_p = scores["LLM + retrieved neighbours"][0]
for other in ["LLM zero-shot", "Copy neighbours' smells (no LLM)", "GNN ensemble (yours)"]:
    m, lo, hi = diff_ci(rag_p, scores[other][0])
    print(f"\nLLM + retrieval minus {other}: {m:+.3f} (95% CI {lo:+.3f} to {hi:+.3f})")

rag_words = Counter(w for s in test.ids for w in dict.fromkeys(done_rag.get(s, [])))
print("\nLLM + retrieval, most-used words:", ", ".join(f"{w} ({c})" for w, c in rag_words.most_common(8)))

Failures: 0 (counted as wrong)

method                               precision@5  recall@5  first pick right
GNN ensemble (yours)                       0.480     0.500              72%
LLM + retrieved neighbours                 0.362     0.360              39%
Copy neighbours' smells (no LLM)           0.417     0.433              64%
LLM zero-shot                              0.193     0.183              38%
Always the 5 most common smells            0.275     0.255              39%

LLM + retrieval minus LLM zero-shot: +0.168 (95% CI +0.149 to +0.189)

LLM + retrieval minus Copy neighbours' smells (no LLM): -0.055 (95% CI -0.072 to -0.038)

LLM + retrieval minus GNN ensemble (yours): -0.118 (95% CI -0.136 to -0.101)

LLM + retrieval, most-used words: fruity (297), green (223), sweet (206), woody (116), herbal (113), fatty (75), fresh (70), nutty (70)


In [11]:
print("LLM + retrieval, first picks:", Counter(done_rag[s][0] for s in test.ids).most_common(5))
print("Copy neighbours, first picks:", Counter(copy_neighbours(i)[0] for i in range(len(test.ids))).most_common(5))

LLM + retrieval, first picks: [('fruity', 123), ('apple', 34), ('alcoholic', 29), ('fatty', 29), ('aldehydic', 23)]
Copy neighbours, first picks: [('fruity', 101), ('floral', 58), ('green', 37), ('sweet', 35), ('woody', 23)]


In [12]:
groups = {"aldehydic": ("aldehyde", Chem.MolFromSmarts("[CX3H1](=O)[#6]")),
          "alcoholic": ("alcohol",  Chem.MolFromSmarts("[CX4][OX2H]"))}
has = lambda s, patt: Chem.MolFromSmiles(s).HasSubstructMatch(patt)
for word, (name, patt) in groups.items():
    firsts = [i for i, s in enumerate(test.ids) if done_rag[s][0] == word]
    share  = np.mean([has(test.ids[i], patt) for i in firsts])
    base   = np.mean([has(s, patt) for s in test.ids])
    right  = np.mean([test.y[i, TASKS.index(word)] == 1 for i in firsts])
    print(f"'{word}' put first {len(firsts)} times: {share:.0%} contain an {name} group "
          f"(vs {base:.0%} of all test molecules); first pick correct {right:.0%} of the time")

'aldehydic' put first 23 times: 48% contain an aldehyde group (vs 5% of all test molecules); first pick correct 17% of the time
'alcoholic' put first 29 times: 34% contain an alcohol group (vs 12% of all test molecules); first pick correct 14% of the time
